## FCLGA GraphTransformer: Nonlinear Demo

This notebook demonstrated the complete FCLGA workflow for CFRP Strain Field Prediction on a Nonlinear Woven-Fabric Composite.

1. **Environment Setup** Verify the conda environment is activated and ready.
2. **Preprocessing pipeline:** 6-step pipeline generating 500 parametric samples.
3. **Training:** Hyperparameter optimized training.  
4. **Testing:** Evaluation on test set. 
5. **Results:** Display of strain field predictions.


## Step 1: Environment Setup

In [13]:
# Setup: Change to project root directory
import os
from pathlib import Path

# Get project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
print(f"✓ Working directory: {PROJECT_ROOT}")

# Verify environment
import sys
print(f"✓ Python: {sys.version.split()[0]}")
print(f"✓ Conda environment: {Path(sys.prefix).name}")

# Quick import check
import torch
import torch_geometric
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ PyTorch Geometric: {torch_geometric.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

✓ Working directory: /home/lpatrign/HybridAttentionGNN
✓ Python: 3.10.19
✓ Conda environment: fclga
✓ PyTorch: 2.9.1+cu128
✓ PyTorch Geometric: 2.7.0
✓ CUDA available: True


## Step 2: Preprocessing Pipeline

In [3]:
# Step 2.1: Generate 500 parametric geometries (~5 min)
!abaqus cae nogui=src/preprocessing/nonlinear/fclga_generate_geometry.py

Abaqus License Manager checked out the following license:
"cae" from Flexnet server abaqus-research.cc.ic.ac.uk
<56 out of 57 licenses remain available>.


In [14]:
# Step 2.2: Run simulations (Dynamic/Explicit)
!python -m src.preprocessing.nonlinear.fclga_run_simulations

Found 500 geometry files to simulate
Input directory: /home/lpatrign/HybridAttentionGNN/data/raw/nonlinear/geometry
Output directory: /home/lpatrign/HybridAttentionGNN/data/raw/nonlinear/simulations

Running 500 simulations with 4 parallel workers

Starting Abaqus job: Plate_nonlinear_0
Starting Abaqus job: Plate_nonlinear_1
Starting Abaqus job: Plate_nonlinear_10
Starting Abaqus job: Plate_nonlinear_100
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_nonlinear_1
Abaqus 2024.HF3
USING MEMORY DOUG-LEA ALLOCATOR
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_nonlinear_100
Abaqus 2024.HF3
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_nonlinear_0
Abaqus 2024.HF3
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_nonlinear_10
Abaqus 2024.HF3
USING MEMORY DOUG-LEA ALLOCATOR
USING MEMORY DOUG-LEA ALLOCATOR
USING MEMORY DOUG-LEA ALLOCATOR
Abaqus License Manager checked out the following licenses:
Abaqus/Exp

In [15]:
# Step 2.3: Extract strains
!abaqus cae nogui=src/preprocessing/nonlinear/fclga_extract_results.py

Abaqus License Manager checked out the following license:
"cae" from Flexnet server abaqus-research.cc.ic.ac.uk
<55 out of 57 licenses remain available>.


In [ ]:
# Step 2.4: Extract features
!python -m src.preprocessing.nonlinear.fclga_extract_features

In [ ]:
# Step 2.5: Build dataset
!python -m src.preprocessing.nonlinear.fclga_build_dataset

In [ ]:
# Step 2.6: Prepare training data
!python -m src.preprocessing.nonlinear.fclga_prepare_training_data

## Step 3: Training

In [ ]:
# Train with hyperparameter optimization

# python -m src.training.fclga_train_model \
#     --material_type nonlinear \
#     --optimize \
#     --optuna_trials 25 \
#     --epochs 50 \
#     --final_epochs 600 \
#     2>&1 | tee notebooks/train_nonlinear.log

# Display the log from the previous run:
from pathlib import Path

log_file = Path("notebooks/train_nonlinear.log")
if log_file.exists():
    log_content = log_file.read_text()
    print(log_content[-20000:])

## Step 4: Testing

Evaluate the trained model on the test set. Use `--training_run` to automatically load the best model and hyperparameters from a training run directory.

**Estimated time:** ~5 minutes

In [ ]:
# Test the model - update the training_run path with your timestamp
!python -m src.evaluation.fclga_test \
    --training_run results/nonlinear/training_nonlinear_20260108_114239 \
    --material_type nonlinear

## Step 5: View Results

Test results are automatically saved as PDFs showing strain field predictions vs ground truth.

**Location:** `results/nonlinear/training_nonlinear_TIMESTAMP/test_sample_X_results.pdf`

Let's display one example result:

In [ ]:
# Display test results as images
from pathlib import Path
from IPython.display import display, Image, HTML
import subprocess

# Update with your training run timestamp
training_run = "results/nonlinear/training_nonlinear_20260108_114239"

# Define all PDFs to display
pdfs_to_display = [
    ("Test Sample Results", Path(training_run) / "test_sample_0_results.pdf"),
    ("Regression Plot", Path(training_run) / "test_sample_0_results_regression.pdf"),
    ("Training Losses", list(Path(training_run).glob("Losses_*.pdf"))[0] if list(Path(training_run).glob("Losses_*.pdf")) else None)
]

def convert_and_display_pdf(pdf_path, title):
    """Convert PDF to PNG and display inline."""
    if pdf_path is None or not pdf_path.exists():
        print(f"⚠️ {title}: PDF not found")
        return False
    
    png_path = pdf_path.with_suffix('.png')
    
    try:
        # Use pdftoppm to convert PDF to PNG
        result = subprocess.run(
            ['pdftoppm', '-png', '-singlefile', '-r', '150', str(pdf_path), str(png_path.with_suffix(''))],
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0 and png_path.exists():
            display(HTML(f"<h3 style='margin-top: 20px; color: #2196F3;'>{title}</h3>"))
            display(Image(filename=str(png_path), width=900))
            return True
        else:
            # Fallback: try using pdf2image
            try:
                from pdf2image import convert_from_path
                images = convert_from_path(str(pdf_path), dpi=150, first_page=1, last_page=1)
                images[0].save(str(png_path), 'PNG')
                display(HTML(f"<h3 style='margin-top: 20px; color: #2196F3;'>{title}</h3>"))
                display(Image(filename=str(png_path), width=900))
                return True
            except ImportError:
                print(f"⚠️ {title}: PDF conversion tools not available")
                print(f"    Location: {pdf_path}")
                return False
    except Exception as e:
        print(f"⚠️ {title}: Could not convert - {e}")
        return False

# Display all PDFs
display(HTML("<h2 style='color: #4CAF50;'>📊 Test Results & Training Metrics</h2>"))
displayed_count = 0
for title, pdf_path in pdfs_to_display:
    if convert_and_display_pdf(pdf_path, title):
        displayed_count += 1

if displayed_count == 0:
    print("\n⚠️ No PDFs found. Run Step 4 (testing) first to generate results.")
    print(f"Expected location: {training_run}")
else:
    print(f"\n✓ Displayed {displayed_count} of {len(pdfs_to_display)} results")